# 💻 Notebook do Aluno — Aula 08: Interfaces com Gradio e Streamlit + deploy com URL pública

**Disciplina:** Prompt Engineering and Artificial Intelligence  
**Instituição:** FIAP — Ciência da Computação · 2026  
**Professor:** Jorge Luiz Gomes  
**Aula 08/14 — Módulo 3: Interfaces, Agentes e Integração**  
**⏱️ 1h40min**  
**🌐 Gradio · ngrok · streaming**  
**🔁 Andaime 45%**  

---

## 🎯 Objetivo da aula

Ao final da aula, o RAG do CKP02 está acessível via URL pública — qualquer pessoa com o link consegue fazer perguntas aos documentos do domínio pelo celular ou computador, sem abrir o Colab.

---

## Como usar este notebook

- Rode as células **na ordem**, de cima para baixo (`Shift+Enter`).
- Complete apenas as partes marcadas com `___` e `👉 LACUNA`.
- Não apague o código já pronto — ele é o andaime do lab.
- Salve sua cópia: **Arquivo > Salvar uma cópia no Drive**.

## 📋 Roteiro do Lab

**Lab — Aula 08 · 2º Semestre**  
### Publicar a primeira URL ao vivo do grupo ★★

*Grupo 3–4 · 25 minutos · Google Colab*

1. Complete as 4 lacunas — retriever, chain com memória (input/history keys), função de chat com streaming e lançamento com share=True.
2. Personalize a interface — título do grupo, 3 exemplos de perguntas reais do domínio como botões de atalho.
3. Publique e compartilhe — copie a URL gerada e poste no grupo da turma. Acesse a URL de outro grupo e faça 2 perguntas.
4. Documente: a interface lembrou o contexto da primeira pergunta na segunda? Teste perguntas de acompanhamento tipo "e sobre isso que você disse..."

> **🎯 Gabarito das lacunas**
>
> Lacuna 1: "k": 3
>
> Lacuna 2: RunnablePassthrough() para "pergunta"; input_messages_key="pergunta"; history_messages_key="chat_history"
>
> Lacuna 3: msg para "pergunta"; sid para "session_id" no config
>
> Lacuna 4: chat para fn; sid para additional_inputs; True para share

---

## 🧩 Notebook Aluno — 45% de lacunas

Complete as lacunas marcadas com `___`.

In [ ]:
!pip install gradio langchain-community langchain-ollama chromadb pymupdf -q

import gradio as gr, uuid, os
from google.colab import userdata
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

In [ ]:
# Reutilizar db do CKP02 (já indexado — só recarregar)
embeddings = OllamaEmbeddings(model="nomic-embed-text")
db         = Chroma(persist_directory="/content/ckp02", embedding_function=embeddings)

# 👉 LACUNA 1: crie o retriever com k=3
retriever = db.as_retriever(search_kwargs={"k":___})

# 👉 LACUNA 2: monte a chain RAG com memória
store = {}
def obter_hist(sid):
    if sid not in store: store[sid] = ChatMessageHistory()
    return store[sid]

chain_base = (
    {"contexto": retriever | RunnableLambda(formatar_contexto),
     "pergunta": ___,
     "chat_history": RunnableLambda(lambda _:"")}
    | prompt_com_hist | llm | StrOutputParser()
)
chain_mem = RunnableWithMessageHistory(
    chain_base, obter_hist,
    input_messages_key=___,
    history_messages_key=___,
)

# 👉 LACUNA 3: função de chat com streaming
def chat(msg, hist, sid):
    parcial = ""
    for chunk in chain_mem.stream({"pergunta":___},
                                   config={"configurable":{"session_id":___}}):
        parcial += chunk
        yield parcial

# 👉 LACUNA 4: monte a interface e publique com share=True
with gr.Blocks() as demo:
    sid = gr.State(lambda:str(uuid.uuid4()))
    gr.Markdown("## 📄 DocMind do Grupo — [Nome do Domínio]")
    gr.ChatInterface(fn=___, type="messages", additional_inputs=[___])
demo.launch(share=___)  # True para URL pública

---

## ✍️ Suas anotações

Registre aqui as observações pedidas no roteiro (qualidade dos resultados, comparações e conclusões do grupo).

## 📚 Referências da aula

- Docs Gradio — ChatInterface, Blocks, streaming e deploy. gradio.app/docs/gradio/chatinterface
- Docs LangChain — RunnableWithMessageHistory para múltiplas sessões. python.langchain.com/docs/how_to/message_history
- Docs ngrok — Túnel HTTP gratuito para desenvolvimento e demos. ngrok.com/docs
- Docs Streamlit — st.chat_message, st.session_state, file_uploader. docs.streamlit.io/develop/api-reference/chat
- Livro Avila, R. D. — Architecting AI Software Systems. Packt, 2025. Cap. 5 — o padrão "Blue and Gold" para deploy seguro de pipelines de IA em produção.
- Livro Russell, S.; Norvig, P. — Inteligência Artificial. 3ª ed. Pearson, 2016. Cap. 2 — Agentes inteligentes: a fundamentação teórica da transição de pipeline para agente que acontece nas próximas aulas.

---

**Próxima Aula — Aula 09** — Agentes de IA — ReAct, tools e function calling
  
O chatbot passa a decidir qual ferramenta usar. RAG, web search e calculadora — autonomamente.

---

*Copyright © 2026 Prof. Jorge Luiz Gomes · FIAP · Todos os direitos reservados.*